# Lab 07: prefix caching and cache hygiene

Cold and warm first token; twenty questions on one system prompt; a timestamp at the top and at the bottom; a sweep over the shared-prefix fraction.

In [ ]:
# Cell 1: the GPU this notebook got. On Kaggle: Settings → Accelerator → GPU T4 x2, then Run all.
import subprocess, sys
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], capture_output=True, text=True)
if out.returncode != 0 or not out.stdout.strip():
    sys.exit("No GPU. Open Settings → Accelerator and choose GPU T4 x2, then run this cell again.")
print(out.stdout.strip())
if "T4" in out.stdout:
    print("Path: Kaggle T4. Lab 10 is read-only on this GPU (no FP8 KV cache); the FP8 checkpoint in Lab 08 runs with unpacked weights.")

In [ ]:
# Cell 2: install the pinned engine. About nine minutes on Kaggle. Kaggle's preinstalled torchaudio is built
# for an older CUDA and breaks the import, so it is removed.
%pip install -q "vllm==0.32.0" 2>&1 | tail -2
%pip uninstall -y -q torchaudio
import vllm, torch
print("vllm", vllm.__version__, "· torch", torch.__version__, "· compute capability", torch.cuda.get_device_capability())

In [ ]:
# Cell 3: helpers used by every lab: start the server in the background, wait for it, talk to it.
import json, re, subprocess, time, urllib.request
MODEL = "Qwen/Qwen3-4B-Instruct-2507"
URL = "http://127.0.0.1:8000"

def serve(extra_args=(), log_path="/kaggle/working/serve.log", timeout_s=900):
    args = ["vllm", "serve", MODEL, "--port", "8000", "--dtype", "half", *extra_args]   # the T4 has no bfloat16
    log = open(log_path, "w")
    t0 = time.time()
    proc = subprocess.Popen(args, stdout=log, stderr=subprocess.STDOUT)
    while time.time() - t0 < timeout_s:
        if proc.poll() is not None:
            raise RuntimeError("vllm serve exited; see " + log_path)
        try:
            urllib.request.urlopen(URL + "/health", timeout=2); return proc, round(time.time() - t0, 1)
        except Exception:
            time.sleep(0.5)
    raise RuntimeError("server not ready in time")

def stop(proc):
    proc.terminate(); proc.wait()

def log_lines(path, patterns):
    text = open(path).read(); out = {}
    for name, pat in patterns.items():
        m = re.search(pat, text)
        if m:
            line = re.sub(r"^\(EngineCore[^)]*\) ", "", m.group(0)); line = re.sub(r"^INFO \d\d-\d\d \d\d:\d\d:\d\d \[[^\]]+\] ", "", line); out[name] = line.strip()
    return out

STARTUP = {"dtype": r"INFO[^\n]*dtype=torch\.[a-z0-9]+", "backend": r"Using [A-Z_]+ attention backend[^\n]*", "weights": r"Loading weights took [^\n]*",
           "model": r"Model loading took [^\n]*", "kv_memory": r"Available KV cache memory: [^\n]*", "kv_size": r"GPU KV cache size: [^\n]*", "graphs": r"Graph capturing finished[^\n]*"}

def chat(messages, **params):
    body = json.dumps({"model": MODEL, "messages": messages, **params}).encode()
    req = urllib.request.Request(URL + "/v1/chat/completions", data=body, headers={"Content-Type": "application/json"})
    return json.loads(urllib.request.urlopen(req, timeout=300).read())
print("helpers ready")

In [ ]:
def stream_chat(messages, **params):
    body = json.dumps({"model": MODEL, "messages": messages, "stream": True, **params}).encode()
    req = urllib.request.Request(URL + "/v1/chat/completions", data=body, headers={"Content-Type": "application/json"})
    t0 = time.time(); first = None; last = None; gaps = []; text = []
    with urllib.request.urlopen(req, timeout=600) as resp:
        for raw in resp:
            line = raw.decode().strip()
            if not line.startswith("data:") or line == "data: [DONE]": continue
            delta = json.loads(line[5:])["choices"][0].get("delta", {}).get("content")
            if not delta: continue
            now = time.time()
            if first is None: first = now - t0
            else: gaps.append(now - last)
            last = now; text.append(delta)
    return first, gaps, "".join(text)

def metrics(names):
    out = {}
    for line in urllib.request.urlopen(URL + "/metrics", timeout=5).read().decode().splitlines():
        if line.startswith("#"): continue
        for n in names:
            if line.startswith(n + "{") or line.startswith(n + " "):
                try: out[n] = float(line.rsplit(" ", 1)[1])
                except ValueError: pass
    return out
print("streaming and metrics helpers ready")

In [ ]:
# Cell 5: cold and warm.
NAMES = ["vllm:prefix_cache_queries_total", "vllm:prefix_cache_hits_total"]
def hits():
    m = metrics(NAMES); return m.get(NAMES[0], 0), m.get(NAMES[1], 0)
proc, ready_s = serve(["--max-model-len", "8192"]); print(f"ready after {ready_s} s")
document = "".join("Paragraph %d of the policy: items may be returned within thirty days in their original packaging with proof of purchase. " % i for i in range(60))
for label in ("cold", "warm"):
    q0, h0 = hits()
    first, _, _ = stream_chat([{"role": "user", "content": document + "\n\nIn one sentence, how long do I have to return an item?"}], max_tokens=30, temperature=0)
    q1, h1 = hits(); print(f"  {label}: first token {first * 1000:5.0f} ms   prefix blocks queried {int(q1 - q0):4d}, hit {int(h1 - h0):4d}")

In [ ]:
# Cell 6: twenty questions on one system prompt, then with a timestamp at the top and at the bottom.
system = "You are the support assistant for a small online shop. " + document[:2400]
q0, h0 = hits(); firsts = []
for i in range(20):
    first, _, _ = stream_chat([{"role": "system", "content": system}, {"role": "user", "content": f"Question {i}: do you ship to country number {i}?"}], max_tokens=20, temperature=0); firsts.append(first)
q1, h1 = hits(); print(f"  hit rate {100 * (h1 - h0) / max(1, q1 - q0):.0f}%   first token: first request {firsts[0] * 1000:.0f} ms, median of the rest {sorted(firsts[1:])[9] * 1000:.0f} ms")
for where in ("top", "bottom"):
    q0, h0 = hits()
    for i in range(20):
        stamp = f"Current time: 2026-10-24T12:00:{i:02d}Z. "; sysm = (stamp + system) if where == "top" else (system + " " + stamp)
        stream_chat([{"role": "system", "content": sysm}, {"role": "user", "content": f"Question {i}: do you ship to country number {i}?"}], max_tokens=20, temperature=0)
    q1, h1 = hits(); print(f"  timestamp at the {where:<6}: hit rate {100 * (h1 - h0) / max(1, q1 - q0):3.0f}%")

In [ ]:
# Cell 7: throughput against the shared-prefix fraction.
def bench_quiet(rate, num_prompts, input_len=1024, output_len=128, max_concurrency=None, prefix_len=0):
    args = ["vllm", "bench", "serve", "--model", MODEL, "--port", "8000", "--dataset-name", "random", "--random-input-len", str(input_len),
            "--random-output-len", str(output_len), "--num-prompts", str(num_prompts), "--request-rate", str(rate),
            "--metric-percentiles", "50,95,99", "--percentile-metrics", "ttft,tpot,itl,e2el", "--ignore-eos", "--seed", "0", "--random-prefix-len", str(prefix_len)]
    if max_concurrency: args += ["--max-concurrency", str(max_concurrency)]
    out = subprocess.run(args, capture_output=True, text=True); text = out.stdout; i = text.find("Serving Benchmark Result")
    res = {}
    for line in text.splitlines():
        m = re.match(r'^(.*?):\s+([0-9.]+)\s*$', line)
        if m: res[m.group(1).strip()] = float(m.group(2))
    return res
for prefix in (0, 256, 512, 768):
    r = bench_quiet(rate=4, num_prompts=48, input_len=1024 - prefix, output_len=128, prefix_len=prefix); g = lambda k: r.get(k, float("nan"))
    print(f"shared {prefix:>4} of 1,024: out tok/s {g('Output token throughput (tok/s)'):7.1f}  TTFT p50 {g('P50 TTFT (ms)'):6.0f} p99 {g('P99 TTFT (ms)'):6.0f} ms")
stop(proc)